# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema available at the following URL.

**Croissant Schema:** [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure mlcroissant is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load dataset metadata and view its description using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access and print metadata summary
metadata = dataset.metadata
print(f"Dataset Name: {getattr(metadata, 'name', None)}\nDescription: {getattr(metadata, 'description', None)}\nVersion: {getattr(metadata, 'version', None)}")

## 2. Data Overview
Inspect all available record sets, their fields, and corresponding `@id`s.

We use the `record_sets` attribute of the metadata object to access all included record sets.

In [ ]:
# List all record sets and their field @id's
record_sets = getattr(metadata, 'record_sets', [])

print("Available record sets and fields (referenced by @id):\n")
record_set_ids = []
all_field_ids = {}
for rs in record_sets:
    rs_id = getattr(rs, '@id', None)
    rs_name = getattr(rs, 'name', rs_id)
    field_objs = getattr(rs, 'fields', [])
    fields_ids = [getattr(f, '@id', None) for f in field_objs]
    record_set_ids.append(rs_id)
    all_field_ids[rs_id] = fields_ids
    print(f"  RecordSet: {rs_name} (@id: {rs_id})")
    print(f"    Fields: {fields_ids}\n")

if not record_set_ids:
    print("No record sets found. The dataset might be defined via distributions; let's inspect those.")
    distributions = getattr(metadata, 'distributions', [])
    print(f"Distributions: {[getattr(d, '@id', None) for d in distributions]}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

**Note:** Some Croissant datasets, including this one, may define all records in a single main record set (often using only one source CSV file). The exact `@id` for the record set can be looked up in the previous section, but if none are present, we'll look for a single distribution and use its implied tabular contents.

In [ ]:
# If record sets are present, use their @ids. Otherwise, try distributions/files.
main_record_set_id = None
if record_set_ids:
    main_record_set_id = record_set_ids[0]  # Default to the first record set
    print(f"Using record set @id: {main_record_set_id}\n")
    
    records = list(dataset.records(record_set=main_record_set_id))
    df = pd.DataFrame(records)
    print(f"Fields in the DataFrame: {df.columns.tolist()}")
    df.head()
else:
    # If not, try to load a DataFrame directly from available file objects via mlcroissant
    distributions = getattr(metadata, 'distributions', [])
    if distributions:
        first_dist_id = getattr(distributions[0], '@id', None)
        print(f"Trying to load records from distribution @id: {first_dist_id}")
        records = list(dataset.records(distribution=first_dist_id))
        df = pd.DataFrame(records)
        print(f"Fields in the DataFrame: {df.columns.tolist()}")
        df.head()
    else:
        print("No record sets or distributions found to extract data.")

## 4. Exploratory Data Analysis (EDA)
Apply several initial data processing steps using the DataFrame for further analysis.

We'll work with a numeric field such as `MW (kDa)` (Molecular Weight) or any other available numeric column, using the column's field `@id` if possible. Please adjust the field below to match an actual numeric field as discovered in the previous extraction step.

In [ ]:
# Select a numeric field for analysis (replace with actual @id if available).
import numpy as np

candidate_numeric_cols = [col for col in df.columns if df[col].dtype in [np.float64, np.float32, np.int64, np.int32]]
if not candidate_numeric_cols:
    # Try to infer numeric columns if dtype is object (e.g. due to missing values or string parsing)
    for col in df.columns:
        # Attempt conversion
        try:
            _ = df[col].astype(float)
            candidate_numeric_cols.append(col)
        except Exception:
            continue
if not candidate_numeric_cols:
    print("No numeric columns found to analyze!")
else:
    numeric_field = candidate_numeric_cols[0]
    print(f"Using numeric field: {numeric_field}")
    # Ensure column is float for processing
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

    threshold = df[numeric_field].mean()  # Example threshold: mean value
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean):")
    print(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    )
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Pick a grouping field: look for a likely categorical column
    possible_group_cols = [col for col in df.columns if col != numeric_field and df[col].nunique() < 15 and df[col].dtype == object]
    group_field = possible_group_cols[0] if possible_group_cols else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"\nGrouped mean {numeric_field} by {group_field}:")
        print(grouped_df.head())

## 5. Visualization
Visualize the data distribution of the selected numeric field and (optionally) group means using matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if candidate_numeric_cols:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if 'group_field' in locals() and group_field:
        means = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
        means.plot(kind='bar', figsize=(10,4), title=f"Mean {numeric_field} by {group_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² dataset of mass spectrometry analysis of extracellular vesicles from stimulated human mast cells using the `mlcroissant` library.

- We demonstrated how to programmatically access metadata, record sets, and extract tabular data by referencing all entities strictly by their `@id` as per Croissant specifications.
- Basic EDA was performed, including filtering by numeric thresholds, normalization, and grouping operations by categorical attributes.
- Data visualizations illustrated the distribution and group means for selected numeric fields.

For further work, consider exploring more complex relationships, integrating external ontologies, or using the Croissant field semantics to drive custom AI/ML pipelines.